In [5]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

TARGET_CRS = "EPSG:5070"

In [8]:
rows = []

# Start browser
driver = webdriver.Chrome()

try:
    # Load page
    driver.get("https://thearp.org/blog/map-archive/")

    # Wait until at least one attachment link is present
    attachments = WebDriverWait(driver, 10).until(
        EC.presence_of_all_elements_located((By.CSS_SELECTOR, "a.attachment-link"))
    )

    print(f"Found {len(attachments)} attachment links.")

    for a in attachments:
        rows.append({
            "text": a.text.strip(),
            "url": a.get_attribute("href")
        })

finally:
    driver.quit()

Found 687 attachment links.


In [11]:
df = pd.DataFrame(rows)

In [ ]:
df["state"] = df["text"].str.split("_").str[0]
df['state'] = df['state'].str.upper()

In [18]:
df['state'].unique()

array(['ALABAMA', 'AL', 'ALASKA', 'AK', 'ARIZONA', 'AZ', 'AR',
       'CALIFORNIA', 'CA', 'COLORADO', 'CO', 'CONNECTICUT', 'CT',
       'DELAWARE', 'DE', 'FL', 'FLORIDA', 'GA', 'GEORGIA', 'HAWAII', 'HI',
       'IDAHO', 'ID', 'ILLINOIS', 'IL', 'INDIANA', 'IN', 'IOWA', 'IA',
       'KANSAS', 'KS', 'KENTUCKY', 'KY', 'LA', 'LOUISIANA', 'ME', 'MAINE',
       'MARYLAND', 'MD', 'MASSACHUSETTS', 'MA', 'MICHIGAN', 'MI', 'MN',
       'MINNESOTA', 'MISSISSIPPI', 'MS', 'MISSOURI', 'MO', 'MONTANA',
       'MT', 'NEBRASKA', 'NE', 'NEVADA', 'NV', 'NEW', 'NH', 'NJ', 'NM',
       'NY', 'NC', 'NORTH', 'ND', 'OH', 'OHIO', 'OKLAHOMA', 'OK',
       'OREGON', 'OR', 'PENNSYLVANIA', 'PA', 'RHODE', 'RI', 'SOUTH', 'SC',
       'SD', 'TENNESSEE', 'TN', 'TX', 'TEXAS', 'UTAH', 'UT', 'VT', 'VA',
       'VIRGINIA', 'WASHINGTON', 'WA', 'WEST', 'WV', 'WISCONSIN', 'WI',
       'WYOMING', 'WY'], dtype=object)

In [19]:
MAP_DF = {
    'ALABAMA' : "AL",
    'ALASKA': "AK", 
    'ARIZONA': "AZ", 
    'CALIFORNIA': "CA", 
    'COLORADO': "CO", 
    'CONNECTICUT': 'CT',
    'DELAWARE': 'DE',
    'FLORIDA': 'FL', 
    'GEORGIA': 'GA',
    'HAWAII': 'HI',
    'IDAHO': 'ID', 
    'ILLINOIS': 'IL', 
    'INDIANA': 'IN', 
    'IOWA': 'IA',
    'KANSAS': 'KS', 
    'KENTUCKY': 'KY', 
    'LOUISIANA': 'LA', 
    'MAINE': 'ME',
    'MARYLAND': 'MD', 
    'MASSACHUSETTS': 'MA', 
    'MICHIGAN': 'MI', 
    'MINNESOTA': 'MN',
    'MISSISSIPPI': 'MS', 
    'MISSOURI': 'MO', 
    'MONTANA': 'MT', 
    'NEBRASKA': 'NE', 
    'NEVADA': 'NV', 
    'OHIO': "OH", 
    'OKLAHOMA': 'OK',
    'OREGON': 'OR', 
    'PENNSYLVANIA': 'PA', 
    'RHODE': 'RI',
    'TENNESSEE': 'TN',
    'TEXAS': 'TX', 
    'UTAH': 'UT',
    'VIRGINIA': "VA", 
    'WASHINGTON': 'WA', 
    'WEST': 'WV', 
    'WISCONSIN': 'WI',
    'WYOMING': 'WY'
}

In [22]:
df["state_abbr"] = df["state"].map(MAP_DF).fillna(df["state"])

In [24]:
df.to_csv("american_redistricting_data.csv")

In [33]:
from pathlib import Path
import requests
import zipfile

In [37]:
output_dir = Path.cwd() / "downloads"
output_dir.mkdir(exist_ok=True)

for _, row in df.iterrows():

    try :

        state_dir = output_dir / row["state_abbr"]
        state_dir.mkdir(parents=True, exist_ok=True)

        filename = row["url"].split("/")[-1]
        zip_path = state_dir / filename

        # print(f"Downloading {filename}")
        r = requests.get(row["url"], stream=True)
        r.raise_for_status()

        with open(zip_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)

        # Create extraction folder with same name as ZIP (minus .zip)
        extract_dir = state_dir / zip_path.stem
        extract_dir.mkdir(exist_ok=True)

        # Extract into that folder
        with zipfile.ZipFile(zip_path, "r") as z:
            z.extractall(extract_dir)

        # Optional: delete the ZIP after extraction
        zip_path.unlink()

    except Exception as e:
        print(f"Failed to extract/convert {filename}: {e}")
        print("Keeping ZIP and continuing...")
        continue

Failed to extract/convert CA_LD_Enacted_Technical_Changes_zaVY5DK.zip: Error -3 while decompressing data: invalid block type
Keeping ZIP and continuing...
Failed to extract/convert ME_LD_Enacted_Technical_Changes.zip: HTTPSConnectionPool(host='thearp.org', port=443): Read timed out. (read timeout=None)
Keeping ZIP and continuing...
Failed to extract/convert MT_LD_Enacted02232023_Technical_Changes.zip: Error -3 while decompressing data: invalid block type
Keeping ZIP and continuing...
Failed to extract/convert ND_US_Cong_2018.zip: HTTPSConnectionPool(host='thearp.org', port=443): Max retries exceeded with url: /documents/862/ND_US_Cong_2018.zip (Caused by ConnectTimeoutError(<HTTPSConnection(host='thearp.org', port=443) at 0x173fa85e350>, 'Connection to thearp.org timed out. (connect timeout=None)'))
Keeping ZIP and continuing...
Failed to extract/convert OH_LD_Enacted05252022.zip: Error -3 while decompressing data: invalid stored block lengths
Keeping ZIP and continuing...
Failed to ex